# Databricks News Digest
Daily email with what changed in the last N days across:
- **Databricks** – release notes, blog, community boards
- **Open source** – Spark, Delta Lake, Unity Catalog, MLflow, Databricks CLI / SDK, dbt-databricks
- **Official docs** – new, updated and removed pages on docs.databricks.com

The email starts with a summary table (`release_date`, `link`, `short_description`, `role`).
SMTP credentials are Unity Catalog secrets (`smtp_user`, `smtp_password`, `mail_to`) in the schema set by the `secret_schema` widget,
created in Catalog Explorer → schema → **Create → Secret**.

In [0]:
%pip install -q feedparser

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## Parameters

In [0]:
dbutils.widgets.text("days_back", "3", "Days back")
dbutils.widgets.dropdown("send_email", "false", ["true", "false"], "Send email")
dbutils.widgets.dropdown("check_docs", "true", ["true", "false"], "Check docs")
dbutils.widgets.text("volume", "workspace.default.databricks_news", "State volume")
dbutils.widgets.text("secret_schema", "workspace.default", "Secrets schema")

DAYS_BACK = int(dbutils.widgets.get("days_back"))
SEND_EMAIL = dbutils.widgets.get("send_email") == "true"
CHECK_DOCS = dbutils.widgets.get("check_docs") == "true"
VOLUME = dbutils.widgets.get("volume")
SECRET_SCHEMA = dbutils.widgets.get("secret_schema")

## Configuration
Edit sources, role keywords and documentation sections here.

In [0]:
COMMUNITY_RSS = "https://community.databricks.com/tmcxu86974/rss/board?board.id="
GITHUB = "https://github.com/"

# (source name, feed url, skip pre-releases)
FEEDS = [
    ("Release notes",                   "https://docs.databricks.com/aws/en/feed.xml", False),
    ("Databricks blog",                 "https://www.databricks.com/feed", False),
    ("Community - Platform updates",    COMMUNITY_RSS + "Product-Platform-Updates", False),
    ("Community - Announcements",       COMMUNITY_RSS + "Announcements", False),
    ("Community - Technical blog",      COMMUNITY_RSS + "technical-blog", False),
    ("Community - MVP articles",        COMMUNITY_RSS + "MVP-ARTICLES", False),
    ("Apache Spark",                    GITHUB + "apache/spark/tags.atom", True),
    ("Delta Lake",                      GITHUB + "delta-io/delta/releases.atom", True),
    ("Unity Catalog OSS",               GITHUB + "unitycatalog/unitycatalog/releases.atom", True),
    ("MLflow",                          GITHUB + "mlflow/mlflow/releases.atom", True),
    ("Databricks CLI",                  GITHUB + "databricks/cli/releases.atom", True),
    ("Databricks SDK for Python",       GITHUB + "databricks/databricks-sdk-py/releases.atom", True),
    ("dbt-databricks",                  GITHUB + "databricks/dbt-databricks/releases.atom", True),
]

# Docs sections checked for updated pages (path after /aws/en/). Empty list = whole site.
DOCS_BASE = "https://docs.databricks.com/aws/en/"
DOCS_SECTIONS = [
    "ldp/", "jobs/", "delta/", "ingestion/", "data-governance/", "sql/", "spark/",
    "dev-tools/", "compute/", "connect/", "query-federation/", "lakehouse-architecture/",
    "machine-learning/", "dashboards/", "genie/", "ai-gateway/", "agents/", "admin/",
]
DOCS_EXCLUDE = ["release-notes/"]  # already covered by the release notes feed
DOCS_WORKERS = 10

# A keyword match in title + description + tags assigns the role; an item can have several roles.
ROLE_KEYWORDS = {
    "Data Engineer": [
        "pipeline", "lakeflow", "ingest", "connector", "job", "workflow", "spark", "delta",
        "streaming", "etl", "dbt", "auto loader", "merge", "runtime", "udf", "bundle", "cli",
        "sdk", "jdbc", "driver", "materialized view", "cdc", "liquid clustering", "photon",
    ],
    "Data Architect": [
        "unity catalog", "governance", "abac", "row filter", "column mask", "lineage",
        "delta sharing", "opensharing", "federation", "iceberg", "metastore", "architecture",
        "private link", "network", "security", "compliance", "encryption", "lakebase", "migration",
    ],
    "Data Analyst": [
        "sql", "dashboard", "ai/bi", "aibi", "genie", "metric view", "excel", "google sheets",
        "warehouse", "visualization", "query", "power bi", "tableau",
    ],
    "ML / AI": [
        "machine learning", "mlflow", "model", "llm", "agent", "generative", "genai",
        "feature store", "feature view", "serving", "vector", "embedding", "gateway", "rag",
        "mcp", "inference",
    ],
    "Platform / Admin": [
        "admin", "workspace", "account", "billing", "budget", "serverless", "compute",
        "cluster", "scim", "service principal", "oauth", "permission",
    ],
}
SOURCE_DEFAULT_ROLE = {
    "Apache Spark": "Data Engineer", "Delta Lake": "Data Engineer", "Databricks CLI": "Data Engineer",
    "Databricks SDK for Python": "Data Engineer", "dbt-databricks": "Data Engineer",
    "Unity Catalog OSS": "Data Architect", "MLflow": "ML / AI",
}
ROLE_COLORS = {
    "Data Engineer": "#1a56db", "Data Architect": "#7e3af2", "Data Analyst": "#0e9f6e",
    "ML / AI": "#c27803", "Platform / Admin": "#6b7280", "General": "#9ca3af",
}

SMTP_HOST, SMTP_PORT = "smtp.gmail.com", 587
USER_AGENT = "Mozilla/5.0 (databricks-news-digest)"

## Helpers

In [0]:
import html
import json
import os
import re
import smtplib
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from email.mime.text import MIMEText

import feedparser
from databricks.sdk import WorkspaceClient


def get_secret(name: str) -> str:
    """Read a Unity Catalog secret (catalog.schema.name) through the Databricks SDK."""
    response = WorkspaceClient().api_client.do(
        "GET", f"/api/2.1/unity-catalog/secrets/{SECRET_SCHEMA}.{name}", query={"include_value": "true"})
    return response["effective_value"]


def clean_text(text: str, limit: int = 300) -> str:
    text = html.unescape(re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", text or ""))).strip()
    return text if len(text) <= limit else text[:limit].rsplit(" ", 1)[0] + "…"


def first_sentence(text: str, limit: int = 160) -> str:
    match = re.match(r"(.+?[.!?])(\s|$)", text)
    return clean_text(match.group(1) if match else text, limit)


def classify(text: str, source: str) -> list:
    text = text.lower()
    roles = [role for role, words in ROLE_KEYWORDS.items()
             if any(re.search(r"(?<![a-z])" + re.escape(w), text) for w in words)]
    return roles or [SOURCE_DEFAULT_ROLE.get(source, "General")]


def make_item(source, title, link, date, summary="", tags=()):
    context = " ".join([title, summary, *tags, link.replace(DOCS_BASE, "").replace("/", " ").replace("-", " ")])
    return {
        "source": source,
        "title": title,
        "link": link,
        "date": date,
        "summary": summary,
        "short": first_sentence(summary) if summary else "",
        "roles": classify(context, source),
    }

## Feeds

In [0]:
PRERELEASE = re.compile(r"(rc\d*|alpha\d*|beta\d*|preview\d*|snapshot|dev\d*)$", re.IGNORECASE)


def entry_date(entry):
    parsed = entry.get("published_parsed") or entry.get("updated_parsed")
    return datetime(*parsed[:6], tzinfo=timezone.utc) if parsed else None


def collect_feeds(cutoff):
    items, errors = [], []
    for source, url, skip_prereleases in FEEDS:
        feed = feedparser.parse(url, agent=USER_AGENT)
        if feed.bozo and not feed.entries:
            errors.append(f"{source}: {feed.get('bozo_exception', 'unreadable feed')}")
            continue

        seen = set()
        for entry in feed.entries:
            title, date = (entry.get("title") or "").strip(), entry_date(entry)
            if not title or not date or date < cutoff or title.lower() in seen:
                continue
            if skip_prereleases and PRERELEASE.search(title):
                continue
            seen.add(title.lower())
            tags = [t["term"] for t in entry.get("tags", []) if t.get("term")]
            items.append(make_item(source, title, entry.get("link", ""), date,
                                   clean_text(entry.get("summary", "")), tags))
    return items, errors

## Documentation
The docs have no RSS feed, so changes are detected in two ways:
- **new / removed pages** – today's sitemap compared with the previous run (state file in the volume)
- **updated pages** – the *Last updated on* date on each page in `DOCS_SECTIONS`; unchanged pages are answered from cache (HTTP 304)

In [0]:
def http_get(url, headers=None, timeout=20):
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, **(headers or {})})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read().decode("utf-8", errors="replace"), response.headers


def parse_last_updated(page):
    pos = page.find("Last updated on")
    if pos < 0:
        return None
    chunk = page[pos:pos + 400]
    match = re.search(r'datetime="([^"]+)"', chunk)
    if match:
        return datetime.fromisoformat(match.group(1).replace("Z", "+00:00")).astimezone(timezone.utc)
    match = re.search(r"([A-Z][a-z]{2,8})\.? (\d{1,2}), (\d{4})", re.sub(r"<[^>]+>", " ", chunk))
    for fmt in ("%b %d %Y", "%B %d %Y"):
        try:
            return datetime.strptime(" ".join(match.groups()), fmt).replace(tzinfo=timezone.utc)
        except (AttributeError, ValueError):
            continue
    return None


def parse_page(page):
    title = re.search(r"<h1[^>]*>(.*?)</h1>", page, re.S) or re.search(r"<title>(.*?)</title>", page, re.S)
    desc = re.search(r'<meta[^>]+name="description"[^>]+content="([^"]*)"', page)
    title = re.sub(r"\s*\|\s*Databricks.*$", "", clean_text(title.group(1))) if title else ""
    return title, clean_text(desc.group(1)) if desc else ""


def title_from_url(url):
    return url.rstrip("/").rsplit("/", 1)[-1].replace("-", " ").capitalize()


def collect_docs(cutoff, state_path):
    now = datetime.now(timezone.utc)
    state = json.load(open(state_path)) if os.path.exists(state_path) else {}
    first_seen = state.get("first_seen", {})   # url -> first time seen in the sitemap
    cache = state.get("pages", {})             # url -> etag / last-modified / parsed fields

    sitemap, _ = http_get(DOCS_BASE + "sitemap.xml", timeout=60)
    urls = sorted({u for u in re.findall(r"<loc>(.*?)</loc>", sitemap) if u.startswith(DOCS_BASE)})

    # New / removed pages. On the first run every page gets an old date so nothing is reported as new.
    first_run = not first_seen
    seen_at = "2000-01-01T00:00:00+00:00" if first_run else now.isoformat()
    removed = sorted(set(first_seen) - set(urls))
    for url in urls:
        first_seen.setdefault(url, seen_at)
    for url in removed:
        first_seen.pop(url)
        cache.pop(url, None)
    new_pages = {u for u, ts in first_seen.items() if datetime.fromisoformat(ts) >= cutoff}

    # Updated pages
    def in_scope(url):
        path = url[len(DOCS_BASE):]
        if any(path.startswith(p) for p in DOCS_EXCLUDE):
            return False
        return not DOCS_SECTIONS or any(path.startswith(p) for p in DOCS_SECTIONS)

    def fetch(url):
        cached = cache.get(url, {})
        headers = {k: v for k, v in (("If-None-Match", cached.get("etag")),
                                     ("If-Modified-Since", cached.get("last_modified"))) if v}
        try:
            page, resp_headers = http_get(url, headers)
            updated = parse_last_updated(page)
            title, desc = parse_page(page)
            cache[url] = {"etag": resp_headers.get("ETag"), "last_modified": resp_headers.get("Last-Modified"),
                          "updated": updated.isoformat() if updated else None, "title": title, "desc": desc}
        except urllib.error.HTTPError as err:
            if err.code != 304:
                return url, err
        except Exception as err:  # noqa: BLE001
            return url, err
        return url, None

    to_check = [u for u in urls if in_scope(u) or u in new_pages]
    print(f"Docs: checking {len(to_check)} pages...")
    with ThreadPoolExecutor(max_workers=DOCS_WORKERS) as pool:
        failed = [u for u, err in pool.map(fetch, to_check) if err]

    items = []
    for url in sorted(new_pages):
        page = cache.get(url, {})
        items.append(make_item("Docs - new pages", page.get("title") or title_from_url(url), url,
                               datetime.fromisoformat(first_seen[url]), page.get("desc", "")))
    for url in to_check:
        page = cache.get(url, {})
        updated = datetime.fromisoformat(page["updated"]) if page.get("updated") else None
        if url not in new_pages and updated and updated >= cutoff:
            items.append(make_item("Docs - updated pages", page["title"] or title_from_url(url), url,
                                   updated, page.get("desc", "")))
    for url in removed:
        items.append(make_item("Docs - removed or renamed pages", title_from_url(url), url, now,
                               "Page is no longer in the sitemap (deleted or renamed)."))

    with open(state_path, "w") as f:
        json.dump({"first_seen": first_seen, "pages": cache, "last_run": now.isoformat()}, f)

    errors = [f"Docs: {len(failed)} of {len(to_check)} pages did not respond"] if failed else []
    if first_run:
        errors.append(f"Docs: first run, saved {len(urls)} pages as baseline; new pages are reported from the next run.")
    return items, errors

## Report and email

In [0]:
def role_badge(role):
    return (f'<span style="background:{ROLE_COLORS.get(role, "#9ca3af")};color:#fff;border-radius:3px;'
            f'padding:1px 6px;margin:1px;font-size:11px;white-space:nowrap">{html.escape(role)}</span>')


def build_report(items, errors, days_back):
    items = sorted(items, key=lambda i: i["date"], reverse=True)
    th = 'style="background:#1b3139;color:#fff;text-align:left;padding:6px 8px;font-size:12px"'
    td = 'style="padding:6px 8px;border-bottom:1px solid #e5e7eb;font-size:12px;vertical-align:top"'

    rows = "".join(
        f'<tr><td {td} nowrap>{i["date"]:%Y-%m-%d}</td>'
        f'<td {td}><a href="{html.escape(i["link"])}" style="color:#1a56db;font-weight:600;text-decoration:none">'
        f'{html.escape(i["title"])}</a><br><span style="color:#888;font-size:11px">{html.escape(i["source"])}</span></td>'
        f'<td {td}>{html.escape(i["short"])}</td>'
        f'<td {td}>{"".join(role_badge(r) for r in i["roles"])}</td></tr>'
        for i in items)

    role_counts = {}
    for i in items:
        for r in i["roles"]:
            role_counts[r] = role_counts.get(r, 0) + 1
    counts = " ".join(f"{role_badge(r)} {n}" for r, n in sorted(role_counts.items(), key=lambda x: -x[1]))
    notes = (f'<p style="font-size:11px;color:#b00">Notes: {html.escape("; ".join(errors))}</p>'
             if errors else "")

    return f"""
<html><body style="font-family:Segoe UI,Arial,sans-serif;max-width:900px;margin:auto;color:#1b1b1b">
<h2 style="color:#FF3621;margin-bottom:4px">Databricks digest – {datetime.now():%Y-%m-%d}</h2>
<p style="color:#666;margin-top:0">{len(items)} updates in the last {days_back} days &nbsp; {counts}</p>
<table cellspacing="0" style="border-collapse:collapse;width:100%">
<tr><th {th}>release_date</th><th {th}>link</th><th {th}>short_description</th><th {th}>role</th></tr>
{rows}</table>{notes}</body></html>"""


def send_email(subject, body):
    user, password = get_secret("smtp_user"), get_secret("smtp_password").replace(" ", "")
    recipients = [r.strip() for r in get_secret("mail_to").split(",") if r.strip()]
    message = MIMEText(body, "html", "utf-8")
    message["Subject"], message["From"], message["To"] = subject, user, ", ".join(recipients)
    with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=30) as smtp:
        smtp.starttls()
        smtp.login(user, password)
        smtp.sendmail(user, recipients, message.as_string())

## Run

In [0]:
catalog, schema, volume = VOLUME.split(".")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume}`")
state_path = f"/Volumes/{catalog}/{schema}/{volume}/docs_state.json"

cutoff = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)
items, errors = collect_feeds(cutoff)
if CHECK_DOCS:
    doc_items, doc_errors = collect_docs(cutoff, state_path)
    items, errors = items + doc_items, errors + doc_errors

report = build_report(items, errors, DAYS_BACK)
print(f"{len(items)} updates since {cutoff:%Y-%m-%d}")
for note in errors:
    print("Note:", note)

if SEND_EMAIL and items:
    send_email(f"[Databricks] {len(items)} updates – last {DAYS_BACK} days", report)
    print("Email sent.")
else:
    displayHTML(report)

Docs: checking 3248 pages...
43 updates since 2026-10-22
Email sent.
